<a href="https://colab.research.google.com/github/cryoTUD/PlotPot/blob/main/PlotPot_sec_mals_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SEC-MALS Plotting Notebook (Google Colab)

Creates publication-quality SEC-MALS chromatograms from ASTRA exports.

**Input format:** Tab-separated ASTRA export with European number format (comma as decimal separator, period as thousands separator).

**Expected column layout:**
```
Columns 0–7 : BSA standard  →  (vol, UV), (vol, dRI), (vol, UV_dup), (vol, LS_raw)
Columns 8–15: Sample         →  (vol, UV), (vol, dRI), (vol, UV_dup), (vol, Mw_Da)
```
The Mw column contains data only inside the integration window set in ASTRA.

**Workflow** — run cells top to bottom:
1. **Dependencies** · **Imports** — run once; collapse when done.
2. **Upload** — file picker appears; select your ASTRA `.txt` export.
3. **Labels & options** — form fields; edit and re-run to update all plots.
4. **Overview** — quick 4-panel sanity check.
5. **Publication plot** — form fields to set elution windows; re-run to iterate.
6. **Save & download** — writes PDF + PNG and downloads to your machine.

In [6]:
#@title Step 1 · Install & verify dependencies { display-mode: "form" }
import importlib, subprocess, sys

_required = ['numpy', 'pandas', 'matplotlib']
_missing  = [p for p in _required if importlib.util.find_spec(p) is None]
if _missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + _missing)
    print(f'Installed: {_missing}')
else:
    print('All dependencies present:', _required)

All dependencies present: ['numpy', 'pandas', 'matplotlib']


In [7]:
#@title Step 2 · Imports & plot style { display-mode: "form" }
import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from pathlib import Path

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size': 10,
    'axes.linewidth': 0.8,
    'xtick.major.width': 0.8,
    'ytick.major.width': 0.8,
    'xtick.direction': 'out',
    'ytick.direction': 'out',
    'pdf.fonttype': 42,
    'svg.fonttype': 'none',
})

In [8]:
#@title Step 3 · Upload data { display-mode: "form" }
# ── Upload your ASTRA export (.txt) ───────────────────────────────────────────
from google.colab import files as _colab_files

print('A file picker will appear below — select your ASTRA export:')
_uploaded = _colab_files.upload()

_fname    = next(iter(_uploaded))
DATA_FILE = Path(_fname)


# ── Parser ────────────────────────────────────────────────────────────────────
def _eu(s: str) -> float:
    """Parse a European-format number ('1.234,56' → 1234.56)."""
    s = str(s).strip()
    if s in ('', 'nan', 'NaN', 'None'):
        return np.nan
    s = s.replace('.', '').replace(',', '.')
    try:
        return float(s)
    except ValueError:
        return np.nan


def load_astra_export(src) -> pd.DataFrame:
    """
    Load a two-sample ASTRA SEC-MALS export file.
    src may be a file path (str/Path) or a file-like object (e.g. io.StringIO).

    Column layout (pairs of volume, signal):
      0–1  : BSA UV        2–3  : BSA dRI
      4–5  : BSA UV dup    6–7  : BSA raw LS  (integration window only)
      8–9  : Sample UV     10–11: Sample dRI
      12–13: Sample UV dup 14–15: Sample Mw in Da  (integration window only)
    """
    col_names = [
        'bsa_uv_vol', 'bsa_uv', 'bsa_ri_vol', 'bsa_ri',
        '_bsa_uv2v',  '_bsa_uv2', 'bsa_ls_vol', 'bsa_ls',
        'dyna_uv_vol', 'dyna_uv', 'dyna_ri_vol', 'dyna_ri',
        '_dyna_uv2v',  '_dyna_uv2', 'dyna_mw_vol', 'dyna_mw_da',
    ]
    if hasattr(src, 'read'):
        fh, _close = src, False
    else:
        fh, _close = open(src, encoding='utf-8'), True
    rows = []
    try:
        for i, line in enumerate(fh):
            if i == 0:
                continue
            parts = line.rstrip('\n').split('\t')
            vals = [_eu(p) for p in parts[0:16]]
            while len(vals) < 16:
                vals.append(np.nan)
            rows.append(vals)
    finally:
        if _close:
            fh.close()
    return pd.DataFrame(rows, columns=col_names)


df = load_astra_export(io.StringIO(_uploaded[_fname].decode('utf-8')))
mw_data = df.dropna(subset=['dyna_mw_vol', 'dyna_mw_da'])

def _rng(col):
    s = df[col].dropna()
    return 'no data' if len(s) == 0 else f'{s.min():.3f} – {s.max():.3f}  (n={len(s):,})'

print(f'Loaded {len(df):,} rows from {_fname}\n')
print(f"BSA  UV  vol : {_rng('bsa_uv_vol')} mL  |  signal : {_rng('bsa_uv')}")
print(f"BSA  dRI vol : {_rng('bsa_ri_vol')} mL  |  signal : {_rng('bsa_ri')}")
print()
print(f"Smpl UV  vol : {_rng('dyna_uv_vol')} mL  |  signal : {_rng('dyna_uv')}")
print(f"Smpl dRI vol : {_rng('dyna_ri_vol')} mL  |  signal : {_rng('dyna_ri')}")
if len(mw_data):
    print(f"Smpl Mw  vol : {mw_data['dyna_mw_vol'].min():.2f} – {mw_data['dyna_mw_vol'].max():.2f} mL  (n={len(mw_data)})")
    print(f"Smpl Mw      : {mw_data['dyna_mw_da'].min()/1e3:.1f} – {mw_data['dyna_mw_da'].max()/1e3:.1f} kDa  (median {mw_data['dyna_mw_da'].median()/1e3:.1f})")
else:
    print('Smpl Mw      : no data in integration window')

A file picker will appear below — select your ASTRA export:


Saving 20251217_dynA.txt to 20251217_dynA (1).txt
Loaded 6,471 rows from 20251217_dynA (1).txt

BSA  UV  vol : 10.001 – 36.491  (n=6,470) mL  |  signal : -0.504 – 1.000  (n=6,470)
BSA  dRI vol : 9.942 – 36.432  (n=6,470) mL  |  signal : -0.004 – 1.000  (n=6,470)

Smpl UV  vol : 0.002 – 26.492  (n=6,470) mL  |  signal : -0.135 – 1.000  (n=6,470)
Smpl dRI vol : -0.057 – 26.433  (n=6,470) mL  |  signal : -0.001 – 1.000  (n=6,470)
Smpl Mw  vol : 12.95 – 13.63 mL  (n=69)
Smpl Mw      : 121.5 – 152.8 kDa  (median 133.0)


In [ ]:
#@title Step 4 · Labels & options { display-mode: "form" }
#@markdown Edit the fields below, then **Run** (Shift+Enter). Leave *Sample label* empty to use the filename stem.

SAMPLE_LABEL = "" #@param {type:"string"}
BSA_LABEL    = "BSA" #@param {type:"string"}
SHOW_BSA     = True #@param {type:"boolean"}

#@markdown ---
#@markdown **Molar mass axis** — log scale, shared by overview and publication plot.
MW_YLIM_MIN  = 1 #@param {type:"number"}
MW_YLIM_MAX  = 10000 #@param {type:"number"}

if not SAMPLE_LABEL.strip():
    SAMPLE_LABEL = DATA_FILE.stem
MW_YLIM = (MW_YLIM_MIN, MW_YLIM_MAX)

print(f'Sample : {SAMPLE_LABEL!r}')
print(f'BSA    : {BSA_LABEL!r}  (show = {SHOW_BSA})')
print(f'Mw axis: {MW_YLIM[0]:,} – {MW_YLIM[1]:,} kDa  (log)')

In [ ]:
#@title Step 5 · Overview plot { display-mode: "form" }
if SHOW_BSA:
    fig, axes = plt.subplots(2, 2, figsize=(12, 7))
    pairs = [
        ('bsa_uv_vol',  'bsa_uv',  f'{BSA_LABEL} UV',    axes[0, 0], 'steelblue'),
        ('bsa_ri_vol',  'bsa_ri',  f'{BSA_LABEL} dRI',   axes[0, 1], 'forestgreen'),
        ('dyna_uv_vol', 'dyna_uv', f'{SAMPLE_LABEL} UV',  axes[1, 0], 'steelblue'),
        ('dyna_ri_vol', 'dyna_ri', f'{SAMPLE_LABEL} dRI', axes[1, 1], 'forestgreen'),
    ]
    ax_sample_uv = axes[1, 0]
else:
    fig, (ax_sample_uv, _ax_ri) = plt.subplots(1, 2, figsize=(10, 3.8))
    pairs = [
        ('dyna_uv_vol', 'dyna_uv', f'{SAMPLE_LABEL} UV',  ax_sample_uv, 'steelblue'),
        ('dyna_ri_vol', 'dyna_ri', f'{SAMPLE_LABEL} dRI', _ax_ri,       'forestgreen'),
    ]

for vcol, scol, title, ax, color in pairs:
    sub = df[[vcol, scol]].dropna()
    ax.plot(sub[vcol], sub[scol], color=color, lw=0.8)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Volume (mL)')
    ax.set_ylabel('Signal')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

ax_mw = ax_sample_uv.twinx()
if len(mw_data):
    ax_mw.scatter(mw_data['dyna_mw_vol'], mw_data['dyna_mw_da'] / 1e3,
                  color='firebrick', s=12, zorder=5, alpha=0.9)
ax_mw.set_yscale('log')
ax_mw.set_ylim(*MW_YLIM)
ax_mw.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:g}'))
ax_mw.set_ylabel('Molar mass (kDa)', color='firebrick')
ax_mw.tick_params(axis='y', labelcolor='firebrick')

fig.suptitle(f'{SAMPLE_LABEL} — overview', fontsize=12)
fig.tight_layout()
plt.show()

In [ ]:
#@title Step 6 · Publication plot { display-mode: "form" }
#@markdown **Elution windows** — set the volume range (mL) to zoom each peak.
SAMPLE_VOL_MIN = 8.0  #@param {type:"number"}
SAMPLE_VOL_MAX = 18.0 #@param {type:"number"}
SAMPLE_VOL_RANGE = (SAMPLE_VOL_MIN, SAMPLE_VOL_MAX)

BSA_VOL_MIN = 21.0 #@param {type:"number"}
BSA_VOL_MAX = 29.0 #@param {type:"number"}
BSA_VOL_RANGE = (BSA_VOL_MIN, BSA_VOL_MAX)

#@markdown **Mw filter** — hide Mw dots where dRI is below this fraction of the peak.
MW_RI_THRESHOLD = 0.10 #@param {type:"slider", min:0.0, max:0.5, step:0.01}

#@markdown ---
#@markdown *Adjust parameters above, then run this cell (Shift+Enter) to regenerate.*

# Colors
C_MW     = '#f0c106'
C_SAMPLE = '#1a4f8a'
C_BSA    = '#888888'


def _slice(df, vcol, scol, vmin, vmax):
    m = (df[vcol] >= vmin) & (df[vcol] <= vmax)
    return df.loc[m, vcol].values, df.loc[m, scol].values


show_bsa = SHOW_BSA and BSA_VOL_RANGE is not None
if show_bsa:
    fig, (ax_d, ax_b) = plt.subplots(1, 2, figsize=(9, 3.8),
                                      gridspec_kw={'wspace': 0.55})
else:
    fig, ax_d = plt.subplots(1, 1, figsize=(4.5, 3.8))

ax_ri = ax_d.twinx()

vd, sd = _slice(df, 'dyna_ri_vol', 'dyna_ri', *SAMPLE_VOL_RANGE)
sd_norm = sd / sd.max() if sd.max() > 0 else sd
ax_ri.plot(vd, sd_norm, color=C_SAMPLE, lw=1.5, label=SAMPLE_LABEL)
ax_ri.set_ylim(-0.06, 1.05)
ax_ri.set_ylabel('Normalized dRI', color='#000000', fontsize=11)
ax_ri.tick_params(axis='y', labelcolor='#000000')

mw_sub = mw_data[
    (mw_data['dyna_mw_vol'] >= SAMPLE_VOL_RANGE[0]) &
    (mw_data['dyna_mw_vol'] <= SAMPLE_VOL_RANGE[1])
].copy()
if len(mw_sub) > 0 and len(vd) > 0:
    ri_at_mw = np.interp(mw_sub['dyna_mw_vol'], vd, sd_norm)
    mw_sub = mw_sub[ri_at_mw >= MW_RI_THRESHOLD]

mw_kda = mw_sub['dyna_mw_da'].values / 1e3
vol_mw = mw_sub['dyna_mw_vol'].values

ax_d.scatter(vol_mw, mw_kda, color=C_MW, s=4, zorder=5, alpha=0.8, label='Molar mass')
if len(mw_kda) > 0:
    mw_median = float(np.median(mw_kda))
    ax_d.axhline(mw_median, color=C_MW, lw=0.8, ls='--', alpha=0.5)
    ax_d.annotate(
        f'{mw_median:.0f} kDa',
        xy=(vol_mw.max(), mw_median),
        xytext=(5, 2), textcoords='offset points',
        color=C_MW, fontsize=9, va='center', ha='left',
    )

ax_d.set_xlim(SAMPLE_VOL_RANGE)
ax_d.set_ylim(*MW_YLIM)
ax_d.set_yscale('log')
ax_d.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:g}'))
ax_d.set_xlabel('Elution volume (mL)', fontsize=11)
ax_d.set_ylabel('Molar mass (kDa)', color='#000000', fontsize=11)
ax_d.tick_params(axis='y', labelcolor='#000000')
ax_d.set_title(SAMPLE_LABEL, fontsize=11, fontweight='bold')

for ax in (ax_d, ax_ri):
    for spine in ax.spines.values():
        spine.set_visible(True)

h1, l1 = ax_ri.get_legend_handles_labels()
h2, l2 = ax_d.get_legend_handles_labels()
ax_d.legend(h1 + h2, l1 + l2, loc='upper right', frameon=False, fontsize=9)

if show_bsa:
    vb, sb = _slice(df, 'bsa_ri_vol', 'bsa_ri', *BSA_VOL_RANGE)
    sb_norm = sb / sb.max() if (len(sb) > 0 and sb.max() > 0) else sb
    ax_b.plot(vb, sb_norm, color=C_BSA, lw=1.5, label=BSA_LABEL)
    ax_b.set_xlim(BSA_VOL_RANGE)
    ax_b.set_ylim(-0.06, 1.05)
    ax_b.set_xlabel('Elution volume (mL)', fontsize=11)
    ax_b.set_ylabel('Normalized dRI', fontsize=11)
    ax_b.set_title(BSA_LABEL, fontsize=11, fontweight='bold')
    for spine in ax_b.spines.values():
        spine.set_visible(True)
    ax_b.legend(loc='upper right', frameon=False, fontsize=9)

fig.tight_layout()
plt.show()

In [ ]:
#@title Step 7 · Save & download { display-mode: "form" }
from google.colab import files as _colab_files

OUTPUT_STEM = DATA_FILE.stem

_outputs = []
for ext in ('pdf', 'png'):
    out = f'/content/{OUTPUT_STEM}_sec_mals.{ext}'
    fig.savefig(out, dpi=300, bbox_inches='tight')
    print(f'Saved → {out}')
    _outputs.append(out)

print('\nStarting downloads...')
for out in _outputs:
    _colab_files.download(out)